# Análisis de integridad de datos sobre un dataset real
### Taller Sesión 2 · Ciber-Recuperación (92-EAN) · Notebook (Colab / Jupyter)

Aplicamos los conceptos de la **Sesión 2** (integridad, entropía, detección de anomalías con
scikit-learn) a un **dataset real de salud** (Alzheimer, Kaggle). Es un caso perfecto porque:

- Es **dato clínico (PHI)** → refuerza la regla del curso: **se analiza local, no se manda a un modelo público**.
- Permite ver la **integridad de datos** en su sentido tabular: nulos, duplicados, clave única,
  reglas de dominio (rangos), y **detección de registros anómalos/corruptos** con IsolationForest.

> Dataset: `rabieelkharoua/alzheimers-disease-dataset` (2149 pacientes × 35 columnas).
> El `DoctorInCharge` ya viene anonimizado como `XXXConfid`: un ejemplo de **de-identificación**.

## 0 · Cargar el dataset (Colab con kagglehub, o archivo local)

In [ ]:
# En Colab:  !pip -q install kagglehub[pandas-datasets]
import os, math, hashlib
import numpy as np, pandas as pd

df = None
# 1) Colab / kagglehub
try:
    import kagglehub
    from kagglehub import KaggleDatasetAdapter
    df = kagglehub.load_dataset(
        KaggleDatasetAdapter.PANDAS,
        "rabieelkharoua/alzheimers-disease-dataset",
        "alzheimers_disease_data.csv",
    )
    print("Cargado vía kagglehub")
except Exception as e:
    # 2) archivo local (si ya lo descargaste)
    for p in ["alzheimers_disease_data.csv", "data/alzheimers_disease_data.csv"]:
        if os.path.exists(p):
            df = pd.read_csv(p); print("Cargado local:", p); break
    if df is None:
        raise SystemExit("No encontré el CSV. Súbelo o usa kagglehub.")

print("Dimensiones:", df.shape)
df.head()

## 1 · Integridad del archivo (huella y entropía)

Igual que en la Sesión 2: una **huella SHA-256** detecta si el archivo fue manipulado (cualquier
cambio de 1 byte cambia el hash), y la **entropía de Shannon** distingue texto de datos cifrados.

In [ ]:
# Guardamos una copia de trabajo para tener una ruta de archivo estable
df.to_csv("dataset_trabajo.csv", index=False)

def sha256(path):
    return hashlib.sha256(open(path, "rb").read()).hexdigest()

def entropia_archivo(path):
    d = open(path, "rb").read()
    if not d: return 0.0
    freq = [0]*256
    for b in d: freq[b] += 1
    n = len(d)
    return -sum((c/n)*math.log2(c/n) for c in freq if c)

print("SHA-256 :", sha256("dataset_trabajo.csv"))
print("Entropía:", round(entropia_archivo("dataset_trabajo.csv"), 2), "bits/byte  (CSV de texto ~3-4)")

# Demostración: 'corromper' = cifrar -> la entropía salta a ~8 (como el ransomware de la Sesión 2)
open("simulado_cifrado.bin", "wb").write(os.urandom(len(open("dataset_trabajo.csv","rb").read())))
print("Entropía de una versión 'cifrada':", round(entropia_archivo("simulado_cifrado.bin"), 2), "bits/byte  -> señal de cifrado")

## 2 · Integridad / calidad de los datos (tabular)

Para un dataset, "integridad" también significa **calidad**: sin nulos inesperados, sin duplicados,
**clave única** (`PatientID`), y valores dentro de **reglas de dominio** (rangos clínicos plausibles,
banderas binarias en {0,1}). Definimos las reglas y medimos violaciones.

In [ ]:
# Reglas de dominio (rangos clínicos plausibles)
RANGES = {"Age": (0, 120), "BMI": (10, 60), "MMSE": (0, 30), "SystolicBP": (70, 220),
          "DiastolicBP": (40, 140), "ADL": (0, 10), "FunctionalAssessment": (0, 10)}
# Columnas binarias (deben ser 0/1): se autodetectan del dataset limpio
BIN = [c for c in df.columns
       if df[c].dtype != object and set(pd.unique(df[c].dropna())) <= {0, 1}]

def reporte_integridad(d):
    r = {}
    r["nulos"]            = int(d.isna().sum().sum())
    r["dup_filas"]        = int(d.duplicated().sum())
    r["dup_PatientID"]    = int(d["PatientID"].duplicated().sum())
    r["fuera_de_rango"]   = int(sum(((d[c] < lo) | (d[c] > hi)).sum()
                                    for c, (lo, hi) in RANGES.items() if c in d))
    r["binario_invalido"] = int(sum((~d[c].isin([0, 1])).sum() for c in BIN if c in d))
    return r

rep_ok = reporte_integridad(df)
print("Columnas binarias detectadas:", len(BIN))
print("Reporte de integridad (dataset original):")
rep_ok

El dataset original debe salir **todo en cero**: es nuestra línea base *conocida-buena*.

## 3 · Detección de registros anómalos (IsolationForest)

El **mismo método de la Sesión 2**, ahora sobre registros clínicos: convertimos cada paciente en un
vector de features numéricas y el modelo **prioriza los más 'raros'** para revisión humana. No dice
"esto está mal", dice **"míralo primero"**.

In [ ]:
from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import StandardScaler

num = df.select_dtypes(include=[np.number]).drop(columns=["PatientID"])
X = StandardScaler().fit_transform(num.fillna(num.median()))
iso = IsolationForest(n_estimators=200, contamination=0.02, random_state=42).fit(X)
sc = iso.score_samples(X)
df["anom_score"] = ((sc.max() - sc) / (sc.max() - sc.min()) * 100).round(1)

print("Top 8 registros anómalos (score 0-100 = más sospechoso):")
cols = [c for c in ["PatientID","Age","BMI","MMSE","SystolicBP","anom_score"] if c in df]
df.sort_values("anom_score", ascending=False)[cols].head(8)

In [ ]:
import matplotlib
matplotlib.use("Agg")  # en Colab/Jupyter normal quítalo
import matplotlib.pyplot as plt
plt.figure(figsize=(7,3))
plt.hist(df["anom_score"], bins=40, color="#0B3FA8")
plt.title("Distribución del score de anomalía")
plt.xlabel("score (0-100)"); plt.ylabel("registros"); plt.tight_layout()
plt.savefig("anom_hist.png", dpi=110); plt.show()
print("La cola derecha = los pocos registros a revisar primero.")

## 4 · Simular corrupción y detectarla

Tomamos una **copia**, le inyectamos errores (edad imposible, BMI negativo, MMSE fuera de rango,
binario inválido, nulos y filas duplicadas) y comprobamos que las **reglas** y el **hash** lo detectan.
Esto es exactamente lo que hace una verificación de integridad antes de restaurar un backup.

In [ ]:
c = df.drop(columns=["anom_score"]).copy()
c.loc[0:2, "Age"]   = 999      # edad imposible
c.loc[3, "BMI"]     = -5       # BMI imposible
c.loc[4, "MMSE"]    = 99       # fuera de rango (0-30)
c.loc[5, "Gender"]  = 2        # binario inválido
c.loc[6, ["MMSE","BMI"]] = np.nan            # nulos
c = pd.concat([c, c.iloc[[10, 11]]], ignore_index=True)  # filas duplicadas
c.to_csv("dataset_corrupto.csv", index=False)

print("Integridad ORIGINAL :", rep_ok)
print("Integridad CORRUPTO :", reporte_integridad(c))
print()
print("SHA-256 original :", sha256("dataset_trabajo.csv")[:32], "...")
print("SHA-256 corrupto :", sha256("dataset_corrupto.csv")[:32], "...  -> DISTINTO = manipulado")

In [ ]:
# Visual: violaciones original vs corrupto
rep_bad = reporte_integridad(c)
labels = list(rep_ok.keys())
import numpy as np
x = np.arange(len(labels)); w = 0.38
plt.figure(figsize=(8,3.2))
plt.bar(x-w/2, [rep_ok[k] for k in labels],  w, label="original", color="#0E7A3C")
plt.bar(x+w/2, [rep_bad[k] for k in labels], w, label="corrupto", color="#B4232A")
plt.xticks(x, labels, rotation=20, ha="right"); plt.legend(); plt.title("Violaciones de integridad")
plt.tight_layout(); plt.savefig("integridad_bar.png", dpi=110); plt.show()

## 5 · La regla forense (PHI): el modelo va a la data, no la data al modelo

Este es **dato de salud (PHI)**. Aunque quieras usar un copiloto de IA para resumir los hallazgos:

- **NO** envíes las filas de pacientes a un modelo público.
- Envía, a lo sumo, un **resumen agregado y sin identificadores** a un modelo **local** (Ollama) o a
  un modelo gestionado en **tu propio tenant**. Nunca los datos crudos.

Mapea a la matriz **datos → backend** del curso y a **OWASP LLM01** (divulgación de información sensible).

In [ ]:
# Resumen AGREGADO y sin PII, apto para pasar a un copiloto LOCAL (no enviamos filas de pacientes)
resumen = {
    "n_registros": int(len(df)),
    "columnas": int(df.shape[1]),
    "integridad_original": rep_ok,
    "integridad_tras_corrupcion": reporte_integridad(c),
    "top_anomalias_scores": df.sort_values("anom_score", ascending=False)["anom_score"].head(5).tolist(),
}
import json
print(json.dumps(resumen, indent=2, ensure_ascii=False))
# En una VM con Ollama:  ollama run llama3.1:8b "Resume estos hallazgos de integridad: <pega el JSON>"

## Cierre

Aplicamos la Sesión 2 a un dataset **real**: huella e integridad de archivo, reglas de calidad,
**detección de anomalías con scikit-learn** y verificación tras corrupción — todo **local**, sin
exponer PHI. Es la misma capacidad que un producto comercial (CyberSense) hace sobre backups, aquí
sobre datos tabulares y con software libre.

*Universidad Ean · Educación Continua · Módulo 92-EAN.*